Dataset structure should be:

<pre>
data
    yes
        ...
    no
        ...

excel
    excel_file.xlsx (doesn't matter what the file is called)
</pre>

In [ ]:
!pip install openpyxl

In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from PIL import Image
import torch.nn.functional as F
from collections import Counter

# Used to process the excel data
import pandas as pd
import shutil

In [ ]:
def excel_to_data(data_dir):
    file_name = os.listdir(data_dir)[0] # First file in the folder
    print(file_name)
    excel_file = pd.ExcelFile(data_dir + '/' + file_name)
    names = [[], []] # names[0] all the "yes", names[1] all the "no"
    scenes = ['40753679', '47333462', '47333473']
    sheets = excel_file.sheet_names
    for s in scenes:
        if s in sheets:
            df = pd.read_excel(data_dir + '/' + file_name, sheet_name = s)
            for i in range(len(df["frame_index"])):
                name = s + '_'
                name += df["query"][i] + '_'
                name += str(df["frame_index"][i]) + '_'
                name += "rendered.png"
                if df["object_present"][i] == True:
                    names[0].append(name)
                else:
                    names[1].append(name)
    print(f"Found {len(names[0])} TRUE")
    print(f"Found {len(names[1])} FALSE")

    # Copy the files from E_grade_pics to data
    copied = [0, 0]
    queries = os.listdir("E_grade_pics")
    for q in queries:
        pic_names = os.listdir("E_grade_pics/" + q)
        for correct in [0, 1]:
            for name in names[correct]:
                if name in pic_names:
                    copied[correct] += 1
                    source_path = "E_grade_pics/" + q + '/' + name
                    if correct == 0:
                        destination_path = "data/yes/" + name
                    else:
                        destination_path = "data/no/" + name
                    shutil.copyfile(source_path, destination_path)
                elif name[-12:] == "rendered.png" and len(name.split('_')) == 4:
                    if name.split('_')[1] == q:
                        print(f"File {name} not found in excel data")
    
    print(f"Copied {copied[0]} files into yes")
    print(f"Copied {copied[1]} files into no")
    return None
    


In [ ]:
def get_queries(data_dir):
    files = []
    for folder in ['yes', 'no']:
        dir = data_dir + '/' + folder
        files += os.listdir(dir)
    
    queries = []
    for f in files:
        q = f.split("_")[1]
        if q not in queries:
            queries.append(q)
    return queries

In [ ]:
def run_one_epoch(model, dl, device, criterion, opt):
    # train
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in dl["train"]:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        opt.step()
        loss_sum += loss.item() * x.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        total    += y.size(0)
    train_loss = loss_sum / total
    train_acc  = correct / total

    # val
    model.eval()
    total = correct = 0
    with torch.no_grad():
        for x, y in dl["val"]:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    val_acc = correct / total
    return train_loss, train_acc, val_acc


In [ ]:
def predict_yes_no(img_path, data_dir="data", ckpt="yesno_resnet.pt"):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Class order from your folder names (e.g., ['no', 'yes'])
    classes = datasets.ImageFolder(data_dir).classes
    num_classes = len(classes)

    # Build the same model and load weights
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.to(device)
    model.eval()

    # Same preprocessing as training
    ORIG_H, ORIG_W = 600, 600
    cut_h = ORIG_H - 85*2
    cut_w = ORIG_W - 15*2
    border_crop = transforms.CenterCrop((cut_h, cut_w))
    norm = ([0.485,0.456,0.406],[0.229,0.224,0.225])
    #norm = ([-0.474, 1.081, 0.717], [1.409, 0.844, 0.981])
    tfm = transforms.Compose([
        border_crop,
        transforms.Resize((224, 224)),
        transforms.ToTensor(), transforms.Normalize(*norm)
    ])

    img = Image.open(img_path).convert("RGB")
    x = tfm(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
        idx = int(probs.argmax().item())
    return classes[idx], float(probs[idx])  # e.g., ('yes', 0.93)

In [ ]:
def check_layers_to_train(layers_to_train, RESNET18_LAYER_NAMES):
    for name in layers_to_train:
        if not name in RESNET18_LAYER_NAMES:
            print(f"LAYER {name} DOES NOT EXIST")


In [ ]:
#excel_to_data("excel") # UNCOMMENT this to load the excel data into "data"

In [ ]:
data_dir = "data"
BATCH_SIZE = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

ORIG_H, ORIG_W = 600, 600
cut_h = ORIG_H - 85*2
cut_w = ORIG_W - 15*2
border_crop = transforms.CenterCrop((cut_h, cut_w))
norm = ([0.485,0.456,0.406],[0.229,0.224,0.225])
#norm = ([-0.474, 1.081, 0.717], [1.409, 0.844, 0.981])
train_tfm = transforms.Compose([
    border_crop,
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(*norm)
])
val_tfm = transforms.Compose([
    border_crop,
    transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(*norm)
])

# Load all images; labels are inferred from folder names ('yes','no')
ds_full = datasets.ImageFolder(data_dir)
print("class_to_idx:", ds_full.class_to_idx)  # {'no': 0, 'yes': 1} (order may vary)

# 80/20 train/val split per class
targets = ds_full.targets
train_idx, val_idx = [], []
for c in set(targets):
    idxs = [i for i,t in enumerate(targets) if t == c]
    random.shuffle(idxs)
    n_train = int(0.8 * len(idxs))
    train_idx += idxs[:n_train]
    val_idx   += idxs[n_train:]

train_ds = Subset(datasets.ImageFolder(data_dir, transform=train_tfm), train_idx)
val_ds = Subset(datasets.ImageFolder(data_dir, transform=val_tfm), val_idx)

dl = {
  "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True),
  "val":   DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True),
}

In [ ]:
# Create model and train it
use_pretrained_net = True
num_classes = len(ds_full.classes) # =2

weights = models.ResNet18_Weights.IMAGENET1K_V1 if use_pretrained_net else None
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

epoch_list = [10, 10]
lr_list = [1e-2, 1e-4]
w_d_list = [1e-5, 1e-2]
layer_list = [["fc"], ["layer4", "fc"]]

best_val_acc = -1.0
for i in range(len(epoch_list)):
    lr = lr_list[i]
    w_d = w_d_list[i]
    epochs = epoch_list[i]

    RESNET18_LAYER_NAMES = ["layer1", "layer2", "layer3", "layer4", "fc"]
    layers_to_train = layer_list[i]
    check_layers_to_train(layers_to_train, RESNET18_LAYER_NAMES)
    for name, param in model.named_parameters():
        top = name.split('.', 1)[0]
        if top in layers_to_train:
            param.requires_grad = True
        else:
            param.requires_grad = False

    print(f"Training layers {layers_to_train}")

    train_targets = torch.tensor([ds_full.targets[i] for i in train_idx])
    class_counts = torch.bincount(train_targets, minlength=num_classes)
    class_weights = (class_counts.sum() / class_counts.clamp_min(1)).float()
    class_weights = class_weights / class_weights.mean()
    class_weights = class_weights.to(device)
    print("class_counts:", class_counts.tolist())
    print("class_weights:", [round(w, 3) for w in class_weights.tolist()])
    
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    
    opt = optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=w_d)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, val_acc = run_one_epoch(model, dl, device, criterion, opt)
        sched.step()
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "yesno_resnet.pt")
        print(f"epoch {epoch}: loss {train_loss:.4f}, train acc {train_acc:.3f}, val acc {val_acc:.3f}"
            + ("  [saved]" if is_best else ""))



In [ ]:
print(predict_yes_no("data/no/40753679_sofa_5_rendered.png"))
print(predict_yes_no("data/yes/40753679_floor_79_rendered.png"))